# 7. Catálogo de Dados

## 7.1 Objetivo do catálogo

Este notebook tem como objetivo documentar os principais ativos de dados construídos ao longo do projeto, registrando sua organização no Lakehouse, finalidade, estrutura e principais atributos.

O catálogo complementa a implementação técnica realizada no Databricks ao fornecer uma visão organizada das tabelas das camadas Bronze, Silver e Gold, facilitando a compreensão dos dados disponíveis e de seu papel no pipeline.

Para as tabelas analíticas da camada Gold, serão documentados também a granularidade, as chaves do modelo dimensional e as medidas utilizadas nas análises.

Ao final, será apresentada a linhagem dos dados, demonstrando o fluxo desde os arquivos brutos disponibilizados pela ANP até as estruturas analíticas utilizadas para responder às perguntas de negócio.

## 7.2 Organização dos dados no Lakehouse

Os dados do projeto foram organizados no catálogo `workspace` do Databricks seguindo a arquitetura Medalhão adotada durante o desenvolvimento.

A estrutura contempla:

- **RAW:** arquivos CSV originais disponibilizados pela ANP, preservados no volume `workspace.default.anp_raw`;
- **Bronze:** tabelas Delta resultantes da ingestão dos arquivos brutos, mantendo sua estrutura próxima à fonte;
- **Silver:** tabela histórica consolidada, com padronização de nomes, tipos, caracteres e nomenclaturas categóricas;
- **Gold:** modelo dimensional em estrela composto por uma tabela fato e quatro dimensões, preparado para consumo analítico.

As tabelas processadas foram distribuídas nos schemas `workspace.bronze`, `workspace.silver` e `workspace.gold`, permitindo separar os dados conforme seu nível de tratamento e finalidade dentro do pipeline.

### 7.2.1 Levantamento das tabelas catalogadas

Para verificar a organização implementada no Lakehouse, serão consultadas as tabelas existentes nos schemas Bronze, Silver e Gold.

Essa consulta permite registrar os ativos persistidos em cada camada e confirmar sua disponibilidade no catálogo do Databricks.

In [0]:
# Lista as tabelas persistidas em cada camada do Lakehouse
for schema in ["bronze", "silver", "gold"]:
    print(f"\nTabelas da camada {schema.upper()}:")
    
    tabelas = spark.sql(
        f"SHOW TABLES IN workspace.{schema}"
    )
    
    display(tabelas)


Tabelas da camada BRONZE:


database,tableName,isTemporary
bronze,producao_mar_2005_2009,false
bronze,producao_mar_2010_2012,false
bronze,producao_mar_2013_2015,false
bronze,producao_mar_2016_2018,false
bronze,producao_mar_2019,false
bronze,producao_mar_2020,false
bronze,producao_mar_2021,false
bronze,producao_mar_2022,false
bronze,producao_mar_2023,false
bronze,producao_mar_2024,false



Tabelas da camada SILVER:


database,tableName,isTemporary
silver,producao_mar,false



Tabelas da camada GOLD:


database,tableName,isTemporary
gold,dim_campo,false
gold,dim_instalacao,false
gold,dim_poco,false
gold,dim_tempo,false
gold,fato_producao,false


#### 7.2.1.1 Resultado do levantamento das tabelas catalogadas

O levantamento confirmou a existência das tabelas persistidas nas três camadas do Lakehouse:

- **Bronze:** 11 tabelas, correspondentes aos diferentes períodos dos arquivos históricos de produção offshore;
- **Silver:** 1 tabela consolidada, `producao_mar`;
- **Gold:** 5 tabelas, sendo quatro dimensões e uma tabela fato.

Ao todo, foram identificadas **17 tabelas** distribuídas entre as camadas Bronze, Silver e Gold.

O resultado confirma que os ativos produzidos durante o pipeline estão persistidos e organizados nos respectivos schemas do catálogo `workspace`.

## 7.3 Catálogo da camada Bronze

A camada Bronze é composta por 11 tabelas Delta provenientes dos arquivos históricos de produção marítima da ANP.

Cada tabela representa um período da série histórica entre 2005 e 2025 e preserva a estrutura dos dados próxima à encontrada nos arquivos de origem. Embora existam diferenças técnicas de representação entre determinados períodos, as tabelas possuem os mesmos **21 atributos lógicos** de produção e injeção.

As tabelas que compõem a camada são:

- `producao_mar_2005_2009`;
- `producao_mar_2010_2012`;
- `producao_mar_2013_2015`;
- `producao_mar_2016_2018`;
- `producao_mar_2019`;
- `producao_mar_2020`;
- `producao_mar_2021`;
- `producao_mar_2022`;
- `producao_mar_2023`;
- `producao_mar_2024`;
- `producao_mar_2025`.

A granularidade observada corresponde a registros mensais de produção e injeção associados a poço, campo e instalação. A consolidação e padronização dessas estruturas são realizadas posteriormente na camada Silver.

### 7.3.1 Dicionário de dados da camada Bronze

O dicionário abaixo apresenta os 21 atributos lógicos presentes nas tabelas da camada Bronze. Os nomes são apresentados conforme a estrutura original da fonte, antes da padronização realizada na camada Silver.

Conforme os metadados disponibilizados pela ANP, a unidade `Mm³` utilizada nos atributos de gás corresponde a **milhares de metros cúbicos**.

| Atributo | Descrição |
|---|---|
| `Ano` | Ano de referência da observação. |
| `Mês/Ano` | Mês e ano de referência da produção. |
| `Estado` | Estado associado ao registro de produção. |
| `Bacia` | Bacia sedimentar associada ao campo produtor. |
| `Campo` | Campo de produção associado ao registro. |
| `Poço` | Identificação do poço. |
| `Ambiente` | Ambiente de produção informado na fonte. |
| `Instalação` | Instalação associada à produção do registro. |
| `Produção de Óleo (m³)` | Volume de óleo produzido, em metros cúbicos. |
| `Produção de Condensado (m³)` | Volume de condensado produzido, em metros cúbicos. |
| `Produção de Gás Associado (Mm³)` | Volume de gás associado produzido, em milhares de metros cúbicos. |
| `Produção de Gás Não Associado (Mm³)` | Volume de gás não associado produzido, em milhares de metros cúbicos. |
| `Produção de Água (m³)` | Volume de água produzido, em metros cúbicos. |
| `Injeção de Gás (Mm³)` | Volume de gás injetado, em milhares de metros cúbicos. |
| `Injeção de Água para Recuperação Secundária (m³)` | Volume de água injetado para recuperação secundária, em metros cúbicos. |
| `Injeção de Água para Descarte (m³)` | Volume de água injetado para descarte, em metros cúbicos. |
| `Injeção de Gás Carbônico (Mm³)` | Volume de gás carbônico injetado, em milhares de metros cúbicos. |
| `Injeção de Nitrogênio (Mm³)` | Volume de nitrogênio injetado, em milhares de metros cúbicos. |
| `Injeção de Vapor de Água (t)` | Quantidade de vapor de água injetada, em toneladas. |
| `Injeção de Polímeros (m³)` | Volume de polímeros injetado, em metros cúbicos. |
| `Injeção de Outros Fluidos (m³)` | Volume de outros fluidos injetado, em metros cúbicos. |

> **Observação:** os tipos físicos não são apresentados como únicos para toda a camada Bronze, pois foram identificadas diferenças de tipagem entre determinados períodos durante a avaliação de qualidade. A padronização dos tipos é realizada na camada Silver.

## 7.4 Catálogo da camada Silver

A camada Silver é representada pela tabela `workspace.silver.producao_mar`, resultante da padronização e consolidação das 11 tabelas históricas da camada Bronze.

A tabela reúne o histórico de produção offshore entre 2005 e 2025 em uma única estrutura com **323.307 registros e 21 atributos**.

Nesta camada foram aplicadas as principais transformações de qualidade e padronização do pipeline, incluindo:

- padronização dos nomes das colunas;
- definição explícita dos tipos de dados;
- conversão do período de referência para o tipo `date`;
- tratamento de caracteres corrompidos;
- padronização de variações de nomenclatura em atributos categóricos;
- preservação dos valores nulos existentes na fonte;
- consolidação dos diferentes períodos históricos.

A tabela Silver mantém a granularidade dos registros provenientes da fonte e constitui a base tratada utilizada para construção do modelo dimensional da camada Gold.

### 7.4.1 Dicionário de dados da camada Silver

| Atributo | Tipo | Descrição |
|---|---|---|
| `ano` | integer | Ano de referência da observação. |
| `mes_ano` | date | Mês e ano de referência da produção. |
| `estado` | string | Estado associado ao registro de produção. |
| `bacia` | string | Bacia sedimentar associada ao campo produtor. |
| `campo` | string | Campo de produção associado ao registro. |
| `poco` | string | Identificação do poço. |
| `ambiente` | string | Ambiente de produção informado na fonte. |
| `instalacao` | string | Instalação associada ao registro. |
| `producao_oleo_m3` | double | Volume de óleo produzido, em metros cúbicos. |
| `producao_condensado_m3` | double | Volume de condensado produzido, em metros cúbicos. |
| `producao_gas_associado_mm3` | double | Volume de gás associado produzido, em milhares de metros cúbicos. |
| `producao_gas_nao_associado_mm3` | double | Volume de gás não associado produzido, em milhares de metros cúbicos. |
| `producao_agua_m3` | double | Volume de água produzido, em metros cúbicos. |
| `injecao_gas_mm3` | double | Volume de gás injetado, em milhares de metros cúbicos. |
| `injecao_agua_recuperacao_secundaria_m3` | double | Volume de água injetado para recuperação secundária, em metros cúbicos. |
| `injecao_agua_descarte_m3` | double | Volume de água injetado para descarte, em metros cúbicos. |
| `injecao_gas_carbonico_mm3` | double | Volume de gás carbônico injetado, em milhares de metros cúbicos. |
| `injecao_nitrogenio_mm3` | double | Volume de nitrogênio injetado, em milhares de metros cúbicos. |
| `injecao_vapor_agua_t` | double | Quantidade de vapor de água injetada, em toneladas. |
| `injecao_polimeros_m3` | double | Volume de polímeros injetados, em metros cúbicos. |
| `injecao_outros_fluidos_m3` | double | Volume de outros fluidos injetados, em metros cúbicos. |

> **Observação:** o sufixo `mm3` foi mantido nos nomes padronizados da camada Silver por derivar da nomenclatura `Mm³` utilizada nos arquivos de origem. Conforme os metadados da ANP, essa unidade corresponde a **milhares de metros cúbicos**. A nomenclatura física das colunas foi preservada para manter a rastreabilidade com a fonte.

### 7.4.2 Validação da estrutura catalogada

Para confirmar que o dicionário de dados apresentado corresponde à estrutura efetivamente persistida no Databricks, será consultado o schema da tabela `workspace.silver.producao_mar`.

A verificação permite comparar os nomes e tipos físicos das colunas com a documentação apresentada no catálogo.

In [0]:
# Exibe a estrutura física da tabela Silver persistida no Databricks
df_silver_catalogo = spark.table("workspace.silver.producao_mar")

df_silver_catalogo.printSchema()

root
 |-- ano: integer (nullable = true)
 |-- mes_ano: date (nullable = true)
 |-- estado: string (nullable = true)
 |-- bacia: string (nullable = true)
 |-- campo: string (nullable = true)
 |-- poco: string (nullable = true)
 |-- ambiente: string (nullable = true)
 |-- instalacao: string (nullable = true)
 |-- producao_oleo_m3: double (nullable = true)
 |-- producao_condensado_m3: double (nullable = true)
 |-- producao_gas_associado_mm3: double (nullable = true)
 |-- producao_gas_nao_associado_mm3: double (nullable = true)
 |-- producao_agua_m3: double (nullable = true)
 |-- injecao_gas_mm3: double (nullable = true)
 |-- injecao_agua_recuperacao_secundaria_m3: double (nullable = true)
 |-- injecao_agua_descarte_m3: double (nullable = true)
 |-- injecao_gas_carbonico_mm3: double (nullable = true)
 |-- injecao_nitrogenio_mm3: double (nullable = true)
 |-- injecao_vapor_agua_t: double (nullable = true)
 |-- injecao_polimeros_m3: double (nullable = true)
 |-- injecao_outros_fluidos_m3: do

### 7.4.2.1 Resultado da validação da estrutura catalogada

A consulta ao schema da tabela `workspace.silver.producao_mar` confirmou a presença dos **21 atributos** documentados no catálogo, com nomes e tipos de dados coerentes com a estrutura definida durante a transformação da camada Silver.

Dessa forma, o dicionário de dados apresentado corresponde à estrutura física efetivamente persistida no Databricks.

## 7.5 Catálogo da camada Gold

A camada Gold contém o modelo dimensional desenvolvido para consumo analítico dos dados de produção offshore.

Foi adotado um **modelo estrela**, composto pela tabela fato `fato_producao` e pelas dimensões `dim_tempo`, `dim_campo`, `dim_poco` e `dim_instalacao`.

As dimensões utilizam chaves substitutas (*surrogate keys*) como chaves primárias, enquanto a tabela fato armazena essas chaves como estrangeiras para estabelecer os relacionamentos do modelo.

A granularidade da tabela fato corresponde a uma observação mensal de produção por combinação de poço, campo e instalação.

As seções seguintes apresentam o dicionário de dados de cada tabela que compõe essa camada.

### 7.5.1 Dimensão Tempo

A tabela `workspace.gold.dim_tempo` representa os períodos mensais existentes na série histórica e permite realizar agregações temporais da produção.

**Granularidade:** um registro por mês.

| Atributo | Tipo | Papel | Descrição |
|---|---|---|---|
| `sk_tempo` | integer | PK | Chave substituta da dimensão Tempo. |
| `mes_ano` | date | Atributo | Mês e ano de referência. |
| `ano` | integer | Atributo | Ano de referência. |
| `mes` | integer | Atributo | Número do mês, de 1 a 12. |
| `nome_mes` | string | Atributo | Nome do mês em português. |

A dimensão contém **252 registros**, correspondentes aos meses de janeiro de 2005 a dezembro de 2025.

### 7.5.2 Dimensão Campo

A tabela `workspace.gold.dim_campo` representa os campos produtores e incorpora os atributos geográficos utilizados nas análises por bacia e estado.

**Granularidade:** um registro por campo.

| Atributo | Tipo | Papel | Descrição |
|---|---|---|---|
| `sk_campo` | integer | PK | Chave substituta da dimensão Campo. |
| `campo` | string | Atributo | Nome do campo produtor. |
| `bacia` | string | Atributo | Bacia sedimentar associada ao campo. |
| `estado` | string | Atributo | Estado associado ao campo. |

A dimensão contém **144 campos distintos**.

### 7.5.3 Dimensão Poço

A tabela `workspace.gold.dim_poco` representa os poços existentes no histórico de produção.

**Granularidade:** um registro por poço.

| Atributo | Tipo | Papel | Descrição |
|---|---|---|---|
| `sk_poco` | integer | PK | Chave substituta da dimensão Poço. |
| `poco` | string | Atributo | Identificação do poço. |

A dimensão contém **2.503 poços distintos**.

### 7.5.4 Dimensão Instalação

A tabela `workspace.gold.dim_instalacao` representa as instalações associadas aos registros de produção.

**Granularidade:** um registro por instalação.

| Atributo | Tipo | Papel | Descrição |
|---|---|---|---|
| `sk_instalacao` | integer | PK | Chave substituta da dimensão Instalação. |
| `instalacao` | string | Atributo | Nome da instalação associada à produção. |

A dimensão contém **231 instalações distintas**. A categoria `Não Informado` é utilizada na camada Gold para permitir o relacionamento dos registros cuja instalação não está identificada na camada Silver.

### 7.5.5 Tabela Fato Produção

A tabela `workspace.gold.fato_producao` concentra as medidas utilizadas nas análises e estabelece o relacionamento com as quatro dimensões do modelo.

**Granularidade:** uma observação mensal de produção por combinação de poço, campo e instalação.

| Atributo | Tipo | Papel | Descrição |
|---|---|---|---|
| `sk_tempo` | integer | FK | Chave estrangeira para `dim_tempo`. |
| `sk_campo` | integer | FK | Chave estrangeira para `dim_campo`. |
| `sk_poco` | integer | FK | Chave estrangeira para `dim_poco`. |
| `sk_instalacao` | integer | FK | Chave estrangeira para `dim_instalacao`. |
| `producao_oleo_mmbbl` | double | Medida | Produção de óleo expressa em milhões de barris (MMbbl), convertida a partir do volume em metros cúbicos disponível na camada Silver. |
| `producao_gas_bilhoes_m3` | double | Medida | Produção total de gás natural expressa em bilhões de metros cúbicos, obtida pela soma dos volumes de gás associado e não associado da camada Silver e posterior conversão de unidade. |

A tabela contém **323.307 registros** e cobre o período de janeiro de 2005 a dezembro de 2025.

As conversões de unidade são realizadas na camada Gold para disponibilizar as medidas em unidades mais adequadas ao consumo analítico, enquanto a camada Silver preserva os valores nas unidades provenientes da fonte.

### 7.5.6 Validação da estrutura catalogada da camada Gold

Para confirmar que os dicionários de dados apresentados correspondem às estruturas efetivamente persistidas no Databricks, serão consultados os schemas físicos das cinco tabelas que compõem a camada Gold.

A verificação permitirá confirmar os nomes e tipos dos atributos, incluindo os tipos efetivamente gerados para as chaves substitutas e estrangeiras do modelo dimensional.

In [0]:
# Exibe o schema físico das tabelas que compõem a camada Gold.
tabelas_gold = [
    "dim_tempo",
    "dim_campo",
    "dim_poco",
    "dim_instalacao",
    "fato_producao"
]

for tabela in tabelas_gold:
    print(f"\nSchema de workspace.gold.{tabela}:")
    spark.table(f"workspace.gold.{tabela}").printSchema()


Schema de workspace.gold.dim_tempo:
root
 |-- sk_tempo: integer (nullable = true)
 |-- mes_ano: date (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- nome_mes: string (nullable = true)


Schema de workspace.gold.dim_campo:
root
 |-- sk_campo: integer (nullable = true)
 |-- campo: string (nullable = true)
 |-- bacia: string (nullable = true)
 |-- estado: string (nullable = true)


Schema de workspace.gold.dim_poco:
root
 |-- sk_poco: integer (nullable = true)
 |-- poco: string (nullable = true)


Schema de workspace.gold.dim_instalacao:
root
 |-- sk_instalacao: integer (nullable = true)
 |-- instalacao: string (nullable = true)


Schema de workspace.gold.fato_producao:
root
 |-- sk_tempo: integer (nullable = true)
 |-- sk_campo: integer (nullable = true)
 |-- sk_poco: integer (nullable = true)
 |-- sk_instalacao: integer (nullable = true)
 |-- producao_oleo_mmbbl: double (nullable = true)
 |-- producao_gas_bilhoes_m3: double (nullable = tru

### 7.5.6.1 Resultado da validação da estrutura catalogada

A consulta aos schemas das cinco tabelas da camada Gold confirmou que os nomes e tipos dos atributos correspondem às estruturas documentadas no catálogo de dados.

As chaves substitutas das dimensões e suas respectivas chaves estrangeiras na tabela fato apresentam tipos compatíveis, permitindo os relacionamentos definidos no modelo dimensional.

Dessa forma, o catálogo apresentado para a camada Gold está coerente com a implementação física realizada no Databricks.

## 7.6 Linhagem dos dados

A linhagem dos dados representa o fluxo percorrido pelas informações desde os arquivos disponibilizados pela ANP até as estruturas analíticas da camada Gold.

O pipeline desenvolvido segue o seguinte fluxo:

**ANP → RAW → Bronze → Silver → Gold → Análises**

1. **Fonte — ANP:** os dados têm origem nos arquivos públicos de produção de petróleo e gás natural por poço disponibilizados pela Agência Nacional do Petróleo, Gás Natural e Biocombustíveis (ANP).

2. **RAW:** os arquivos CSV originais são armazenados no volume `workspace.default.anp_raw`, preservando os dados recebidos da fonte sem alterações.

3. **Bronze:** os arquivos são ingeridos e persistidos como tabelas Delta no schema `workspace.bronze`. Nessa etapa são realizados os tratamentos necessários para a correta leitura dos diferentes formatos encontrados nos arquivos históricos, mantendo a estrutura próxima à fonte.

4. **Silver:** as 11 tabelas históricas são padronizadas e consolidadas na tabela `workspace.silver.producao_mar`. Nessa camada são tratados nomes e tipos de colunas, representações numéricas, caracteres corrompidos e variações de nomenclatura, preservando os registros e os valores nulos da fonte.

5. **Gold:** a tabela Silver é transformada em um modelo dimensional no schema `workspace.gold`, composto pelas dimensões `dim_tempo`, `dim_campo`, `dim_poco` e `dim_instalacao` e pela tabela `fato_producao`.

6. **Análises:** as tabelas da camada Gold constituem a fonte das análises destinadas a responder às perguntas de negócio do projeto.

Essa estrutura permite acompanhar a transformação dos dados desde sua origem até o consumo analítico, mantendo separadas as etapas de ingestão, tratamento, modelagem e análise.

## 7.7 Conclusão do catálogo de dados

O catálogo de dados documentou os principais ativos construídos ao longo do pipeline, abrangendo as camadas Bronze, Silver e Gold do Lakehouse.

Foram registrados a finalidade, a estrutura e os principais atributos das tabelas, além da granularidade, chaves e medidas do modelo dimensional da camada Gold. As consultas aos schemas físicos permitiram confirmar a correspondência entre a documentação e as estruturas efetivamente persistidas no Databricks.

A linhagem apresentada também consolidou o fluxo dos dados desde os arquivos públicos da ANP até o modelo dimensional utilizado para consumo analítico.

Com isso, os dados encontram-se não apenas processados e modelados, mas também catalogados e documentados, facilitando sua compreensão, rastreabilidade e utilização nas análises do projeto.